# 00 - Introduction: theme, methods and objectives

**Owner:** Member 1
**Task:** Introduce the project: what a recommender system on a user-item graph is, what exposure bias is and why it matters for evaluation, why we compare BPR-MF with LightGCN, and what the experiments are meant to answer.
**Inputs:** none
**Outputs:** none (text only)

The two datasets are described in notebook 01.

## 1. Theme

**Does LightGCN's advantage survive unbiased evaluation?**

**Why short-video feeds.** Short-video feeds are no longer limited to dedicated short-video apps: e-commerce and social platforms now embed them too. In a feed, the app pushes one item at a time and the user rarely searches, so the recommender decides almost everything the user sees. How a feed recommender is evaluated before it is deployed therefore matters more here than in search-driven products.

A recommender system predicts which items a user will interact with next. With *implicit feedback* we only observe positive events (a user clicked a video, a user watched it twice); everything else is unknown, not negative. The task is **top-K recommendation**: for each user, rank all items the user has not interacted with and return the best K (we use K = 20).

The observed interactions form a **bipartite graph**: users on one side, items on the other, an edge for every positive interaction.

```
users        items
 u1 ─────────  i1
    ╲
 u2 ───╲─────  i2        u1 and u2 both liked i2,
        ╲                 so i3 (liked by u2) is a candidate for u1:
 u3 ─────╲───  i3        the path u1 - i2 - u2 - i3 has length 3.
```

Collaborative filtering uses only this graph: no video content, no user profile.

## 2. The problem: exposure bias

A user can only click a video that was shown to them, and on a real platform the platform's own recommender decides what is shown. The logged data therefore over-represents videos the old recommender already liked to show, which are mostly popular ones. A test set cut from the same logs has the same bias, so it rewards a new model for agreeing with the old recommender. A model that simply recommends popular items can look good for that reason alone.

Both of our datasets come from Kuaishou, a short-video app, and both contain a part that the platform recommender did **not** choose:

| Dataset | Biased part (train and ordinary test) | Unbiased part (second test set) |
|---|---|---|
| KuaiRand-Pure | Videos shown by the platform recommender | Videos inserted into the feed **at random** |
| KuaiRec | Videos shown by the platform recommender | A block of 1,411 users and 3,327 videos where **every** user saw **every** video |

We train every model once on the biased data and score the same model on both test sets.

In a feed almost every interaction is on a video the recommender pushed, so exposure bias is at its strongest. That makes short-video data the natural place to test it.

**Scope.** Both datasets come from one short-video platform. Our results apply to feed-style recommendation; we do not claim they carry over to other settings such as product search.

## 3. The two methods and why we chose them

Both models give every user and every item a vector (embedding) and score a pair by the dot product. Both are trained with the same BPR loss, which asks that an observed item scores higher than a randomly sampled unobserved one.

| | BPR-MF | LightGCN |
|---|---|---|
| Idea | Matrix factorization: learn one vector per user and per item | Same vectors, then smooth them over the user-item graph |
| Final embedding | The learned vector itself | Mean of the vector and its 1, 2, ..., K-hop neighbourhood averages |
| Uses the graph | Only through the loss (one edge at a time) | Explicitly, through K rounds of neighbour averaging |
| Learned parameters | User and item embeddings | The same embeddings, nothing else |
| Extra cost | None | K sparse matrix products per training step |
| Reference | Rendle et al., UAI 2009 | He et al., SIGIR 2020 |

LightGCN propagation (no weight matrices, no nonlinearity):

$$e_u^{(k+1)} = \sum_{i \in N(u)} \frac{1}{\sqrt{|N(u)|}\sqrt{|N(i)|}}\, e_i^{(k)}, \qquad e_u = \frac{1}{K+1} \sum_{k=0}^{K} e_u^{(k)}$$

**Why this pair.**
- LightGCN with K = 0 layers *is* BPR-MF. The two models have the same parameters, loss and negative sampling, so any difference in the results comes from graph propagation alone.
- Propagation spreads information along edges, and popular items have the most edges. It is therefore a reasonable suspicion that part of LightGCN's usual advantage is a popularity effect, which is exactly what a biased test set rewards. That makes this pair a natural one for the exposure-bias question.
- Both train on a CPU at the size of our datasets.

We add a **most-popular** baseline (recommend the items with the most training interactions to everyone). It needs no training and tells us how much of each model's score could be earned by popularity alone.

## 4. Objectives

1. **Headline:** compare most-popular, BPR-MF and LightGCN on the biased and the unbiased test of both datasets, under identical splits, seeds, tuning budget and evaluation. Does LightGCN's advantage over BPR-MF hold on the unbiased tests, and how large is each model's margin over most-popular?
2. **Explain the result** by breaking recall down by item popularity and by user degree on both tests (notebook 07).
3. **Understand LightGCN** through a layer study (notebook 04) and an ablation of its components (notebook 05).
4. **Check the role of density** by removing training interactions (notebook 08).
5. **Show success and failure cases** and propose an improvement based on them (notebook 07).
6. **Practical takeaway:** state what an offline evaluation should include before a graph recommender is deployed in a feed (an unbiased test where one exists, and a most-popular baseline).

| Notebook | Question it answers |
|---|---|
| 01 | What is in the data, how were the unbiased parts collected, and what graph do the models see? |
| 03, 04 | Which settings are best for each model (chosen on validation)? How many layers? |
| 05 | Which LightGCN component matters? |
| 06 | Headline: which model is better on each test, and by how much relative to noise and to most-popular? |
| 07 | For which items and users does each model gain or lose on the unbiased test? |
| 08 | Does the advantage of the graph depend on density? |